# D3 — Paths are messages / Los caminos son mensajes

🇬🇧 Here is the bridge between D1 (over-squashing) and D2 (counting paths). In a GNN, information flows along
edges. After `g` layers, a message from node `i` reaches node `j` **once for every length-`g` path** between
them. So the path count `n_g(i,j)` from D2 is exactly **how many copies of `i`'s information arrive at `j`** —
and all those copies must fit in `j`'s single fixed-size vector. That is over-squashing, now made precise.

🇪🇸 Aquí está el puente entre D1 (over-squashing) y D2 (conteo de caminos). En una GNN, la información fluye
por las aristas. Tras `g` capas, un mensaje del nodo `i` llega al nodo `j` **una vez por cada camino de
longitud `g`** entre ellos. Así que el conteo `n_g(i,j)` de D2 es exactamente **cuántas copias de la
información de `i` llegan a `j`** — y todas esas copias deben caber en el único vector de tamaño fijo de `j`.
Eso es el over-squashing, ahora hecho preciso.

<img src="img/02_path_is_message.svg" width="720" alt="Each distinct path from i to j delivers one copy of i's message; the path count equals the number of copies arriving."/>

<sub>🇬🇧 Four distinct paths `i→j` of length 3 → four copies of `i`'s message arrive at `j`. · 🇪🇸 Cuatro caminos distintos `i→j` de longitud 3 → cuatro copias del mensaje de `i` llegan a `j`.</sub>

## Watch the messages pile up / Mira cómo se acumulan los mensajes

🇬🇧 We take the bottleneck graph and, for growing depth, count the messages arriving at the target. The same
explosion from D1 — but now we read it as *message multiplicity*.

🇪🇸 Tomamos el grafo cuello de botella y, para profundidad creciente, contamos los mensajes que llegan al
objetivo. La misma explosión de D1 — pero ahora la leemos como *multiplicidad de mensajes*.

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash.ideal_bridge import walk_operators

K, M = 5, 4
depths = [1,2,3,4]; msgs = []
for d in depths:
    data = make_bottleneck_graph(K, M, d, torch.Generator().manual_seed(0))
    raw, _ = walk_operators(data.edge_index.numpy(), data.num_nodes, max_length=d+1)
    t = int(data.root_mask.nonzero()[0])
    msgs.append(int(raw[d][:, t].sum()))
for d, m in zip(depths, msgs):
    print(f'depth {d}: {m:5d} messages arrive at the target  (= K·M^d = {K}·{M}^{d})')
plt.figure(figsize=(5.5,3.2))
plt.semilogy(depths, msgs, 'o-', color='#dc2626')
plt.xlabel('distance (depth)'); plt.ylabel('messages at target (log)')
plt.title('message multiplicity explodes with distance'); plt.grid(alpha=.3); plt.show()

## A subtlety: redundant vs. distinct messages / Una sutileza: mensajes redundantes vs. distintos

🇬🇧 Not all of those messages carry *new* information. In our bottleneck, the middle nodes are interchangeable,
so many of the paths are **functionally identical** — they carry the same content by different routes. A
natural idea (which the thesis explores) is to *merge* equivalent paths using a **quotient** `kQ/I`, so the
count drops from `K·M^d` down to just `K`.

🇪🇸 No todos esos mensajes llevan información *nueva*. En nuestro cuello de botella, los nodos del medio son
intercambiables, así que muchos caminos son **funcionalmente idénticos** — llevan el mismo contenido por rutas
distintas. Una idea natural (que la tesis explora) es *fusionar* caminos equivalentes con un **cociente**
`kQ/I`, para que el conteo baje de `K·M^d` a solo `K`.

In [ ]:
# raw count vs. de-duplicated (quotient) count / conteo crudo vs. sin duplicados (cociente)
data = make_bottleneck_graph(K, M, 3, torch.Generator().manual_seed(0))
raw, eff = walk_operators(data.edge_index.numpy(), data.num_nodes, max_length=4)
t = int(data.root_mask.nonzero()[0])
print(f'raw messages       (A^g):  {int(raw[3][:,t].sum())}')
print(f'de-duplicated (kQ/I, M_g):  {int(eff[3][:,t].sum())}   <-- collapses to K = {K}')

## The plot twist / El giro inesperado

🇬🇧 You might guess that merging redundant messages *fixes* over-squashing. **We tested it — and it does not.**
On a task where the target must know *which* source matters, throwing away the multiplicity throws away signal:
the de-duplicating model actually does **worse**. (See notebook `04` in the parent folder for the numbers.)

So merging is the wrong move. The *right* move — the one that works — is in **D4**: keep all the paths, but let
the network **learn how much to weight each one**. That is attention.

🇪🇸 Podrías suponer que fusionar mensajes redundantes *arregla* el over-squashing. **Lo probamos — y no.** En
una tarea donde el objetivo debe saber *cuál* fuente importa, descartar la multiplicidad descarta señal: el
modelo que de-duplica lo hace **peor**. (Ver el cuaderno `04` en la carpeta superior para los números.)

Así que fusionar es el movimiento equivocado. El movimiento *correcto* — el que funciona — está en **D4**:
conservar todos los caminos, pero dejar que la red **aprenda cuánto pesar cada uno**. Eso es atención.